<a href="https://colab.research.google.com/github/sravyaabburi1/dataviz-exercises-sravyaabburi/blob/main/lecture03_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


In [ ]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [2]:
import pandas as pd
import plotly.graph_objects as go

# Load dataset
df = pd.read_csv('/content/co2_emissions.csv')

# Filter dataset to Asia only
asia = df[df['Region'] == 'Asia']

# Choose country to highlight
highlight_country = 'China'

# Create figure
fig = go.Figure()

# Loop through every country
for country in asia['Country'].unique():

    # Filter data for one country
    country_df = asia[asia['Country'] == country]

    # Highlight selected country
    if country == highlight_country:

        fig.add_trace(
            go.Scatter(
                x=country_df['Year'],
                y=country_df['CO2_Mt'],
                mode='lines',
                name=country,
                line=dict(
                    color='#D62728',
                    width=3
                )
            )
        )

        # Add direct label at end of line
        fig.add_annotation(
            x=country_df['Year'].max(),
            y=country_df['CO2_Mt'].iloc[-1],
            text=country,
            showarrow=False,
            xshift=12,
            font=dict(
                family='Arial',
                size=12,
                color='#D62728'
            )
        )

    # Add all other countries in grey
    else:

        fig.add_trace(
            go.Scatter(
                x=country_df['Year'],
                y=country_df['CO2_Mt'],
                mode='lines',
                line=dict(
                    color='#DDDDDD',
                    width=1
                ),
                hoverinfo='skip',
                showlegend=False
            )
        )

# Update layout
fig.update_layout(
    title={
        'text': 'China’s CO₂ emissions surged far faster than other Asian countries since 2000',
        'x': 0.5,
        'xanchor': 'center'
    },

    font=dict(
        family='Arial',
        size=12
    ),

    plot_bgcolor='white',
    paper_bgcolor='white',

    width=1000,
    height=600,

    showlegend=False,

    margin=dict(
        l=60,
        r=80,
        t=80,
        b=60
    )
)

# Clean x-axis
fig.update_xaxes(
    title='Year',
    showgrid=False,
    showline=False,
    zeroline=False
)

# Clean y-axis
fig.update_yaxes(
    title='CO₂ Emissions (Mt)',
    showgrid=True,
    gridcolor='#EEEEEE',
    showline=False,
    zeroline=False
)

# Show chart
fig.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [3]:
import pandas as pd
import plotly.graph_objects as go

# Load dataset
df = pd.read_csv('/content/co2_emissions.csv')

# Calculate regional average emissions by year
regional_avg = (
    df.groupby(['Region', 'Year'])['CO2_Mt']
    .mean()
    .reset_index()
)

# Keep only 2000 and 2022
slope_df = regional_avg[
    regional_avg['Year'].isin([2000, 2022])
]

# Create figure
fig = go.Figure()

# Loop through each region
for region in slope_df['Region'].unique():

    region_data = slope_df[slope_df['Region'] == region]

    # Get values
    y_start = region_data[region_data['Year'] == 2000]['CO2_Mt'].values[0]
    y_end = region_data[region_data['Year'] == 2022]['CO2_Mt'].values[0]

    # Colour based on increase/decrease
    if y_end > y_start:
        line_color = '#D62728'   # Red for increase
    else:
        line_color = '#1F77B4'   # Blue for decrease

    # Add slope line
    fig.add_trace(
        go.Scatter(
            x=[2000, 2022],
            y=[y_start, y_end],
            mode='lines',
            line=dict(
                color=line_color,
                width=3
            ),
            showlegend=False,
            hoverinfo='skip'
        )
    )

    # Left-side label (2000)
    fig.add_annotation(
        x=2000,
        y=y_start,
        text=f"{region}: {y_start:.1f}",
        showarrow=False,
        xshift=-45,
        font=dict(
            family='Arial',
            size=11,
            color='black'
        )
    )

    # Right-side label (2022)
    fig.add_annotation(
        x=2022,
        y=y_end,
        text=f"{region}: {y_end:.1f}",
        showarrow=False,
        xshift=45,
        font=dict(
            family='Arial',
            size=11,
            color='black'
        )
    )

# Layout styling
fig.update_layout(
    title={
        'text': 'Asia and Africa saw the largest increases in average CO₂ emissions since 2000',
        'x': 0.5,
        'xanchor': 'center'
    },

    font=dict(
        family='Arial',
        size=12
    ),

    plot_bgcolor='white',
    paper_bgcolor='white',

    width=900,
    height=600,

    margin=dict(
        l=100,
        r=100,
        t=80,
        b=60
    )
)

# Clean x-axis
fig.update_xaxes(
    tickvals=[2000, 2022],
    showgrid=False,
    showline=False,
    zeroline=False
)

# Remove y-axis tick labels
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    showline=False,
    zeroline=False,
    title=''
)

# Show figure
fig.show()
